# Module 23: Deployment Basics — Solutions

Complete solutions to all exercises.

In [ ]:
import json
import time
import numpy as np
from pathlib import Path
from typing import List, Optional, Dict, Any
from dataclasses import dataclass

### Solution 1: Dockerfile

In [ ]:
dockerfile = """
FROM python:3.10-slim AS builder

WORKDIR /app
COPY requirements.txt .
RUN pip install --user --no-cache-dir -r requirements.txt && \\
    rm -rf /root/.cache/pip

FROM python:3.10-slim

WORKDIR /app
COPY --from=builder /root/.local /root/.local
COPY app/ app/
COPY models/ models/

ENV PATH=/root/.local/bin:$PATH \\
    PYTHONUNBUFFERED=1 \\
    PYTHONDONTWRITEBYTECODE=1

RUN groupadd -r mluser && useradd -r -g mluser mluser && \\
    chown -R mluser:mluser /app
USER mluser

EXPOSE 8000

HEALTHCHECK --interval=30s --timeout=3s --start-period=10s --retries=3 \\
    CMD python -c "import urllib.request; resp=urllib.request.urlopen('http://localhost:8000/health'); assert resp.status == 200"

CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]
"""
print("Solution 1 — Multi-stage Dockerfile with healthcheck")

### Solution 2: FastAPI Prediction Endpoint

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

app = FastAPI(title="House Price Predictor", version="1.0.0")

MODEL_LOADED = True
START_TIME = time.time()


class PredictRequest(BaseModel):
    features: List[float] = Field(..., min_length=1, max_length=100,
                                 description="Feature vector")

class PredictResponse(BaseModel):
    prediction: float
    model_version: str = "v1.0"
    processing_time_ms: float

class HealthResponse(BaseModel):
    status: str
    model_loaded: bool
    uptime_seconds: float


@app.get("/", response_model=Dict[str, str])
def root():
    return {"name": "House Price Predictor", "version": "1.0.0", "status": "running"}

@app.get("/health", response_model=HealthResponse)
def health():
    return HealthResponse(
        status="healthy",
        model_loaded=MODEL_LOADED,
        uptime_seconds=time.time() - START_TIME,
    )

@app.post("/predict", response_model=PredictResponse)
def predict(req: PredictRequest):
    if not MODEL_LOADED:
        raise HTTPException(status_code=503, detail="Model not loaded")
    t0 = time.time()
    features = np.array(req.features).reshape(1, -1)
    prediction = float(np.sum(features) * 1000)  # dummy model
    elapsed = (time.time() - t0) * 1000
    return PredictResponse(
        prediction=prediction,
        processing_time_ms=round(elapsed, 2),
    )

print("Solution 2 — FastAPI app defined")
print("Endpoints: GET /, GET /health, POST /predict")

### Solution 3: API Tests with TestClient

In [ ]:
from fastapi.testclient import TestClient

client = TestClient(app)


def test_root():
    response = client.get("/")
    assert response.status_code == 200
    assert "name" in response.json()
    assert "version" in response.json()

def test_health():
    response = client.get("/health")
    assert response.status_code == 200
    data = response.json()
    assert data["status"] == "healthy"
    assert data["model_loaded"] == True

def test_predict():
    response = client.post("/predict", json={"features": [1.0, 2.0, 3.0]})
    assert response.status_code == 200
    data = response.json()
    assert "prediction" in data
    assert "processing_time_ms" in data

def test_predict_invalid_input():
    response = client.post("/predict", json={"features": []})
    assert response.status_code == 422

def test_predict_missing_field():
    response = client.post("/predict", json={})
    assert response.status_code == 422


test_root()
test_health()
test_predict()
test_predict_invalid_input()
test_predict_missing_field()
print("Solution 3 — All API tests PASSED")

### Solution 4: docker-compose.yml

In [ ]:
compose = """
version: "3.8"

services:
  api:
    build:
      context: .
      dockerfile: Dockerfile
    ports:
      - "8000:8000"
    environment:
      - MODEL_PATH=/app/models/model.pkl
      - LOG_LEVEL=INFO
      - REDIS_URL=redis://redis:6379/0
    volumes:
      - ./models:/app/models:ro
      - ./logs:/app/logs
    depends_on:
      redis:
        condition: service_started
    healthcheck:
      test: ["CMD", "python", "-c", "import urllib.request; resp=urllib.request.urlopen('http://localhost:8000/health'); assert resp.status == 200"]
      interval: 30s
      timeout: 10s
      retries: 3
      start_period: 15s
    restart: unless-stopped
    networks:
      - ml_network

  redis:
    image: redis:7-alpine
    ports:
      - "6379:6379"
    volumes:
      - redis_data:/data
    restart: unless-stopped
    networks:
      - ml_network

volumes:
  redis_data:

networks:
  ml_network:
    driver: bridge
"""
print("Solution 4 — docker-compose with API + Redis + health check")

### Solution 5: Model Versioning

In [ ]:
class ModelManager:
    """Manage multiple model versions with caching."""

    def __init__(self, models_dir="models"):
        self.models_dir = Path(models_dir)
        self._models: Dict[str, Any] = {}
        self._load_available_versions()

    def _load_available_versions(self):
        self.versions = {}
        for path in self.models_dir.glob("*.pkl"):
            version = path.stem.replace("model_", "").replace("model-", "")
            if version == "model":
                version = "v1"
            self.versions[version] = str(path)

    def load_model(self, version="latest"):
        if version == "latest":
            version = sorted(self.versions.keys())[-1] if self.versions else "v1"
        if version not in self._models:
            if version not in self.versions:
                raise ValueError(f"Version '{version}' not found")
            import joblib
            self._models[version] = joblib.load(self.versions[version])
        return self._models[version]

    def list_versions(self):
        return list(self.versions.keys())


print("Solution 5 — ModelManager class with versioning")
print("  Supports: load_model('v1'), load_model('latest'), list_versions()")

### Solution 6: CI/CD Pipeline

In [ ]:
deploy_workflow = """
name: Deploy ML API

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  test-and-validate:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - uses: actions/setup-python@v4
        with:
          python-version: "3.10"
      - name: Install dependencies
        run: |
          pip install -r requirements.txt
          pip install pytest pytest-cov flake8
      - name: Lint
        run: flake8 app/ tests/
      - name: Run tests
        run: pytest tests/ --cov=app/ --cov-fail-under=80 -v
      - name: Validate model
        run: python scripts/validate_model.py

  build-and-deploy:
    needs: test-and-validate
    if: github.ref == 'refs/heads/main'
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Build Docker image
        run: docker build -t ml-api:${{ github.sha }} .
      - name: Log in to GHCR
        run: echo "${{ secrets.GITHUB_TOKEN }}" | docker login ghcr.io -u ${{ github.actor }} --password-stdin
      - name: Push image
        run: |
          docker tag ml-api:${{ github.sha }} ghcr.io/${{ github.repository }}:latest
          docker push ghcr.io/${{ github.repository }}:latest
      - name: Deploy via SSH
        uses: appleboy/ssh-action@v0.1.5
        with:
          host: ${{ secrets.DEPLOY_HOST }}
          username: ${{ secrets.DEPLOY_USER }}
          key: ${{ secrets.DEPLOY_KEY }}
          script: |
            cd /opt/ml-api
            docker-compose pull api
            docker-compose up -d --force-recreate api
            docker system prune -f
"""
print("Solution 6 — GitHub Actions CI/CD pipeline with validation, build, push, and deploy")

### Solution 7: Drift Monitoring Script

In [ ]:
from scipy.stats import ks_2samp


class DriftMonitor:
    """Monitor drift between reference and production data."""

    def __init__(self, reference_data: np.ndarray, feature_names: List[str],
                 threshold: float = 0.05):
        self.reference = reference_data
        self.feature_names = feature_names
        self.threshold = threshold

    def check_drift(self, production_data: np.ndarray) -> Dict:
        results = {"drift_detected": False, "features": {}, "timestamp": time.time()}
        for i, name in enumerate(self.feature_names):
            stat, pval = ks_2samp(self.reference[:, i], production_data[:, i])
            drifted = bool(pval < self.threshold)
            results["features"][name] = {
                "ks_statistic": round(stat, 4),
                "p_value": round(pval, 6),
                "drift_detected": drifted,
            }
            if drifted:
                results["drift_detected"] = True
        return results

    def generate_report(self, drift_results: Dict) -> str:
        report_lines = ["=" * 50]
        report_lines.append("DRIFT MONITORING REPORT")
        report_lines.append(f"Timestamp: {time.ctime(drift_results['timestamp'])}")
        report_lines.append(f"Overall drift: {'DETECTED' if drift_results['drift_detected'] else 'NONE'}")
        report_lines.append("")
        for name, info in drift_results["features"].items():
            status = "DRIFT" if info["drift_detected"] else "OK"
            report_lines.append(f"  {name:20s} KS={info['ks_statistic']:.4f} p={info['p_value']:.6f} [{status}]")
        report_lines.append("=" * 50)
        return "\n".join(report_lines)


# Test
ref = np.random.normal(0, 1, (1000, 3))
prod = np.random.normal(0.2, 1.1, (1000, 3))
monitor = DriftMonitor(ref, ["feature_a", "feature_b", "feature_c"])
results = monitor.check_drift(prod)
print(monitor.generate_report(results))

### Solution 8: Performance Benchmark

In [ ]:
def benchmark_api(url: str, n_requests: int = 100, batch_size: int = 1):
    """Benchmark API latency and throughput."""
    latencies = []
    errors = 0

    for _ in range(n_requests):
        payload = {"features": list(np.random.randn(batch_size).tolist())}
        t0 = time.time()
        try:
            import requests
            resp = requests.post(f"{url}/predict", json=payload, timeout=5)
            if resp.status_code == 200:
                latencies.append((time.time() - t0) * 1000)
            else:
                errors += 1
        except Exception:
            errors += 1

    if not latencies:
        return {"error": "All requests failed", "errors": errors}

    latencies.sort()
    return {
        "n_requests": n_requests,
        "batch_size": batch_size,
        "successful": len(latencies),
        "errors": errors,
        "p50_ms": round(latencies[len(latencies) // 2], 2),
        "p95_ms": round(latencies[int(len(latencies) * 0.95)], 2),
        "p99_ms": round(latencies[int(len(latencies) * 0.99)], 2),
        "throughput_req_per_sec": round(len(latencies) / (sum(latencies) / 1000), 2),
    }


print("Solution 8 — Benchmark function defined")
print("Returns: p50, p95, p99 latency and throughput")